# 02 · Exploratory Data Analysis

**Purpose:** Retain the original univariate, bivariate, multivariate, geographic, and time-pattern exploration while making each result easier to interpret.

This notebook preserves the substantive code and analytical sequence from the original completed project. Changes are limited to portable file paths, organization, and explanatory markdown.


## Reproducible setup

The project-relative paths below replace the original Google Colab mount so the notebook runs consistently in VS Code, Jupyter, or GitHub Codespaces.


In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA = PROJECT_ROOT / "data" / "raw" / "cdc_brfss_nutrition_activity_obesity_2011_2024.csv"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed" / "obesity_analysis_ready.csv"

# Each analysis notebook is independently runnable from the project root or notebooks folder.
clean_df = pd.read_csv(PROCESSED_DATA, low_memory=False)
data = clean_df.copy()
print(f"Loaded {len(clean_df):,} analysis-ready rows covering {clean_df.YearStart.min()}–{clean_df.YearStart.max()}.")


## 3. EDA

### 3.1 Basic EDA

#### 3.1.1 UNIVARIATE ANALYSIS

**Why this analysis matters:** The distribution reveals typical prevalence levels, spread, skew, and potential extreme values before group comparisons.


In [ ]:
#Obesity Distribution

plt.figure(figsize=(6,4))
plt.hist(clean_df["Data_Value"], bins=100)
plt.xlabel("Adult Obesity Percentage")
plt.ylabel("Frequency")
plt.title("Distribution of Adult Obesity Rates")
plt.show()


**What this step does:** Executes the next stage of the original analysis and preserves its result for the interpretation that follows.


In [ ]:
clean_df["Data_Value"].describe()


#### 3.1.2 BIVARIATE ANALYSIS

**Why this analysis matters:** Location averages highlight where prevention resources and deeper investigation may be most valuable.


In [ ]:
import matplotlib.pyplot as plt

# Obesity by State

state_avg = clean_df.groupby("LocationDesc")["Data_Value"].mean().sort_values(ascending=True)

plt.figure(figsize=(10, 12))  # Increased figure size for better readability
state_avg.plot(kind="barh")
plt.xlabel("Average Obesity %")
plt.ylabel("State") # Added y-axis label
plt.title("Average Adult Obesity by State")
plt.grid(axis='x', linestyle='--', alpha=0.7) # Added grid lines
plt.tight_layout() # Adjust layout to prevent labels from being cut off
plt.show()

**What this code examines:** Compares survey sample size with prevalence estimates to check whether estimate level appears related to coverage.


In [ ]:
# Obesity vs Sample Size

plt.figure(figsize=(6,4))
plt.scatter(clean_df["Sample_Size"], clean_df["Data_Value"], alpha=0.7)
plt.xlabel("Sample Size")
plt.ylabel("Obesity Percentage")
plt.title("Obesity vs Sample Size")
plt.show()


**What this step does:** Executes the next stage of the original analysis and preserves its result for the interpretation that follows.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Group by education level and calculate the mean obesity percentage
obesity_by_education = clean_df[clean_df['StratificationCategory1'] == 'Education'].groupby('Stratification1')['Data_Value'].mean().sort_values(ascending=False)

# Create the bar chart
plt.figure(figsize=(10, 6))
sns.barplot(x=obesity_by_education.index, y=obesity_by_education.values, palette='viridis')
plt.xlabel("Education Level")
plt.ylabel("Average Obesity Percentage")
plt.title("Average Adult Obesity Percentage by Education Level")
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

**Why this step matters:** Confidence-interval width indicates estimate precision and helps prevent over-interpreting uncertain subgroup estimates.


In [ ]:
# Confidence Interval Analysis

clean_df["CI_Width"] = clean_df["High_Confidence_Limit"] - clean_df["Low_Confidence_Limit"]

plt.figure(figsize=(6,4))
plt.hist(clean_df["CI_Width"], bins=10)
plt.xlabel("Confidence Interval Width")
plt.title("Uncertainty in Obesity Estimates")
plt.show()

#### 3.1.3 MULTIVARIATE / PATTERN ANALYSIS

**Why this analysis matters:** The time series shows whether prevalence is stable, improving, or worsening across the study period.


In [ ]:
# Obesity Distribution by Stratification
# Analyze the trend of obesity over time
yearly_trend = clean_df.groupby("YearStart")["Data_Value"].mean()

# Create a plot for the yearly trend
plt.figure(figsize=(8, 5))  # Adjusted figure size for better aspect ratio
yearly_trend.plot(marker="o", linestyle='-', color='skyblue') # Added marker, linestyle, and color
plt.xlabel("Year") # Added x-axis label
plt.ylabel("Average Obesity %")
plt.title("Trend of Adult Obesity Over Time (Average Across Locations/Stratifications)") # More descriptive title
plt.grid(True, linestyle='--', alpha=0.6) # Added grid for better readability
plt.xticks(rotation=45) # Rotate x-axis labels if they overlap
plt.tight_layout() # Adjust layout to prevent labels from being cut off
plt.show()

**What this code does:** Quantifies linear relationships among prevalence, sample size, and uncertainty. Correlation is interpreted as association, not causation.


In [ ]:
# Correlation

clean_df[["Data_Value", "Sample_Size", "CI_Width"]].corr()


State-Level Trend Comparison (Top vs Bottom States)

**What this step does:** Executes the next stage of the original analysis and preserves its result for the interpretation that follows.


In [ ]:
top_states = (
    clean_df.groupby("LocationDesc")["Data_Value"]
    .mean()
    .sort_values(ascending=False)
    .head(3)
    .index
)

low_states = (
    clean_df.groupby("LocationDesc")["Data_Value"]
    .mean()
    .sort_values()
    .head(3)
    .index
)

plt.figure(figsize=(10, 6)) # Increased figure size

for state in list(top_states) + list(low_states):
    state_data = clean_df[clean_df["LocationDesc"] == state].groupby('YearStart')['Data_Value'].mean().reset_index()
    plt.plot(state_data["YearStart"], state_data["Data_Value"], label=state, marker='o') # Added marker

plt.xlabel("Year")
plt.ylabel("Obesity Percentage")
plt.title("Obesity Trends: Highest vs Lowest Prevalence States")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7) # Added grid
plt.tight_layout() # Added tight_layout
plt.show()

Variability Trend (Stability vs Volatility)

**What this step does:** Executes the next stage of the original analysis and preserves its result for the interpretation that follows.


In [ ]:
yearly_variability = clean_df.groupby("YearStart")["Data_Value"].std()

plt.figure(figsize=(8,5)) # Adjusted figure size for better aspect ratio
plt.plot(yearly_variability.index, yearly_variability.values, marker='o', linestyle='-', color='coral') # Added marker, linestyle, and color
plt.xlabel("Year")
plt.ylabel("Standard Deviation of Obesity Rates") # More descriptive y-axis label
plt.title("Yearly Variability in Obesity Rates Across States")
plt.grid(True, linestyle='--', alpha=0.7) # Added grid for better readability
plt.tight_layout() # Adjust layout to prevent labels from being cut off
plt.show()

- A higher standard deviation (a higher point on the curve) indicates greater variability, meaning there's a wider range of obesity rates among states in that particular year.

- A lower standard deviation (a lower point on the curve) suggests less variability, meaning the obesity rates across states are more clustered around the average for that year.

**What this step does:** Executes the next stage of the original analysis and preserves its result for the interpretation that follows.


In [ ]:
#Sample Size Over Time

sample_trend = clean_df.groupby("YearStart")["Sample_Size"].mean()

plt.figure(figsize=(7,4))
plt.plot(sample_trend.index, sample_trend.values, marker='o')
plt.xlabel("Year")
plt.ylabel("Average Sample Size")
plt.title("Trend of Survey Sample Size Over Time")
plt.show()


**What this code does:** Reshapes the long survey data into a comparison-friendly table without changing the underlying estimates.


In [ ]:
# Heatmap Trend (State-Year Pattern)

pivot_df = clean_df.pivot_table(
    values="Data_Value",
    index="LocationDesc",
    columns="YearStart"
)

plt.figure(figsize=(14, 16)) # Increased figure size to accommodate more labels
sns.heatmap(pivot_df, cmap="Reds", linewidths=.5, linecolor='white', # Added linewidths and linecolor
            cbar_kws={'label': 'Obesity Prevalence (%)'}) # Added color bar label
plt.title("Heatmap of Obesity Prevalence Across States and Years", fontsize=16) # Increased title font size
plt.xlabel("Year", fontsize=12) # Added x-axis label font size
plt.ylabel("State", fontsize=12) # Added y-axis label font size
plt.xticks(rotation=45) # Rotate x-axis labels for better readability
plt.tight_layout() # Adjust layout to prevent labels from being cut off
plt.show()